# Basic Data Engineering
### A One-Day Intensive: 2–3 Hour Lesson + 4 Hours Hands-On
**Powered by Google Colaboratory**

Build working pipelines end-to-end: files & formats → SQL → **ETL** → API ingestion → data quality → orchestration concepts. **Core libraries:** `pandas`, `pyarrow` (Parquet), `sqlite3`, `requests`, `re`, `json`, `logging`.

---
**How to use this notebook**
1. Run cells top to bottom (`Shift + Enter` runs a cell).
2. The **Setup** section generates all dummy datasets (CSV, TXT, Parquet, and a mock API JSON) — no uploads needed.
3. Lesson sections (Parts 1 onward) are the **2–3 hour instructor-led portion**.
4. The **Hands-On Lab** section at the end is the **4-hour guided exercise portion** with a capstone.

---
> © **Data Engineering Pilipinas (DEP). All rights reserved.**
> This course material was created by and is the property of Data Engineering Pilipinas. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.

## ⚙️ Setup — Generate the Dummy Datasets
Run the cell below **once** at the start of the session. It creates four practice datasets in a `data/` folder:

| File | Format | Simulates |
|---|---|---|
| `sales_data.csv` | CSV | Retail sales orders (intentionally *messy*: missing values, duplicates, mixed date formats, bad emails) |
| `server_logs.txt` | TXT | Raw web-server logs (for **regex** practice) |
| `transactions.parquet` | Parquet | Bank transactions (columnar format) |
| `customers_api.json` | JSON | A saved **API response** with customer records |

In [1]:
# =============================================================
# SETUP — Run this cell first! It generates all dummy datasets.
# Creates: sales_data.csv, server_logs.txt, transactions.parquet,
#          customers_api.json  (a saved mock API response)
# =============================================================
import pandas as pd
import numpy as np
import json, random, os

np.random.seed(42)
random.seed(42)
os.makedirs("data", exist_ok=True)

# ---------- 1. CSV: retail sales data (intentionally messy) ----------
n = 500
regions  = ["NCR", "Cebu", "Davao", "Iloilo", "Baguio"]
products = ["Rice 25kg", "Cooking Oil 1L", "Instant Coffee",
            "Laundry Soap", "Canned Sardines", "Sugar 1kg"]
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")

df = pd.DataFrame({
    "order_id":   [f"ORD-{i:05d}" for i in range(1, n + 1)],
    "order_date": np.random.choice(dates, n),
    "region":     np.random.choice(regions, n),
    "product":    np.random.choice(products, n),
    "quantity":   np.random.randint(1, 20, n),
    "unit_price": np.round(np.random.uniform(20, 1500, n), 2),
    "customer_email": [f"customer{i}@{random.choice(['gmail.com','yahoo.com','outlook.ph'])}" for i in range(n)],
    "phone":      [f"09{random.randint(100000000, 999999999)}" for _ in range(n)],
})
df["total_amount"] = (df["quantity"] * df["unit_price"]).round(2)

# Inject "messiness" so we can practice cleaning
dirty = df.copy()
dirty.loc[dirty.sample(25, random_state=1).index, "unit_price"] = np.nan
dirty.loc[dirty.sample(15, random_state=2).index, "region"] = " ncr "
dirty.loc[dirty.sample(10, random_state=3).index, "customer_email"] = "invalid-email"
dirty = pd.concat([dirty, dirty.sample(12, random_state=4)])   # duplicates
dirty["order_date"] = dirty["order_date"].astype(str)
mixed_idx = dirty.sample(20, random_state=5).index
dirty.loc[mixed_idx, "order_date"] = dirty.loc[mixed_idx, "order_date"].str.replace("-", "/")
dirty.to_csv("data/sales_data.csv", index=False)

# ---------- 2. TXT: server log file (for regex practice) ----------
levels    = ["INFO", "WARNING", "ERROR", "DEBUG"]
endpoints = ["/api/orders", "/api/customers", "/api/products", "/login", "/checkout"]
lines = []
for i in range(300):
    ts  = pd.Timestamp("2025-06-01") + pd.Timedelta(seconds=int(np.random.uniform(0, 86400 * 30)))
    lvl = random.choices(levels, weights=[60, 15, 10, 15])[0]
    ip  = f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    ep  = random.choice(endpoints)
    ms  = random.randint(5, 2500)
    status = random.choices([200, 201, 404, 500, 403], weights=[70, 10, 8, 7, 5])[0]
    lines.append(f"{ts.strftime('%Y-%m-%d %H:%M:%S')} [{lvl}] client={ip} "
                 f"endpoint={ep} status={status} response_time={ms}ms")
with open("data/server_logs.txt", "w") as f:
    f.write("\n".join(sorted(lines)))

# ---------- 3. Parquet: bank-style transactions ----------
m = 800
tx = pd.DataFrame({
    "transaction_id":   [f"TXN{i:07d}" for i in range(1, m + 1)],
    "account_id":       [f"ACC{random.randint(1000,1099)}" for _ in range(m)],
    "transaction_type": np.random.choice(["deposit","withdrawal","transfer","payment"], m, p=[.3,.3,.2,.2]),
    "amount":           np.round(np.random.lognormal(mean=7, sigma=1.2, size=m), 2),
    "channel":          np.random.choice(["mobile_app","atm","branch","online"], m),
    "timestamp":        pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.uniform(0,365,m), unit="D"),
    "is_flagged":       np.random.choice([True, False], m, p=[.04,.96]),
})
tx.to_parquet("data/transactions.parquet", index=False)

# ---------- 4. Mock API response saved as JSON ----------
first = ["Maria","Jose","Juan","Ana","Carlo","Liza","Ramon","Grace","Paolo","Nina"]
last  = ["Santos","Reyes","Cruz","Garcia","Torres","Flores","Ramos","Mendoza"]
customers = []
for i in range(1, 61):
    fn, ln = random.choice(first), random.choice(last)
    customers.append({
        "customer_id": i,
        "name": f"{fn} {ln}",
        "email": f"{fn.lower()}.{ln.lower()}{i}@example.com",
        "city": random.choice(["Quezon City","Makati","Cebu City","Davao City","Taguig"]),
        "signup_date": str((pd.Timestamp("2024-01-01") + pd.Timedelta(days=random.randint(0,700))).date()),
        "loyalty_points": random.randint(0, 5000),
        "is_active": random.random() > 0.15,
    })
with open("data/customers_api.json", "w") as f:
    json.dump({"status": "success", "count": len(customers), "data": customers}, f, indent=2)

print("✅ Setup complete! Files created in ./data/:")
for fn in sorted(os.listdir("data")):
    print("   -", fn)

✅ Setup complete! Files created in ./data/:
   - customers_api.json
   - sales_clean.csv
   - sales_data.csv
   - server_logs.txt
   - transactions.parquet


## Part 1 — Python Essentials (≈30 min)
Python is the *lingua franca* of data work. We only need a focused subset: **variables, collections, control flow, functions**, and the habit of writing small, reusable pieces of logic.

In [ ]:
# Variables & data types
store_name = "DEP Sari-Sari Store"   # str
daily_sales = 15750.50               # float
num_customers = 42                   # int
is_open = True                       # bool

print(type(store_name), type(daily_sales), type(num_customers), type(is_open))
print(f"{store_name} earned ₱{daily_sales:,.2f} from {num_customers} customers today.")

In [ ]:
# Collections: list, tuple, dict, set
products = ["rice", "oil", "coffee", "soap"]            # list  — ordered, mutable
location = (14.6760, 121.0437)                          # tuple — ordered, immutable (e.g., lat/lon)
prices   = {"rice": 52.0, "oil": 85.5, "coffee": 7.5}   # dict  — key/value lookup
regions  = {"NCR", "Cebu", "NCR", "Davao"}              # set   — unique values only

print("3rd product:", products[2])
print("Price of oil:", prices["oil"])
print("Unique regions:", regions)

In [ ]:
# Control flow: if / for / while
sales = [1200, 3400, 800, 5600, 2300]

total = 0
for amount in sales:
    total += amount
    if amount > 3000:
        print(f"  High-value sale: ₱{amount:,}")

print(f"Total: ₱{total:,}")
print(f"Average: ₱{total/len(sales):,.2f}")

# List comprehension — the "pythonic" one-liner loop
vat_inclusive = [round(s * 1.12, 2) for s in sales]
print("With 12% VAT:", vat_inclusive)

In [ ]:
# Functions + error handling — the building blocks of reusable data code
def compute_discount(amount, customer_type="regular"):
    """Return the discounted amount based on customer type."""
    rates = {"regular": 0.0, "senior": 0.20, "pwd": 0.20, "member": 0.05}
    if customer_type not in rates:
        raise ValueError(f"Unknown customer type: {customer_type}")
    return round(amount * (1 - rates[customer_type]), 2)

print(compute_discount(1000))             # 1000.0
print(compute_discount(1000, "senior"))   # 800.0

# try/except keeps pipelines and analyses from crashing on bad input
try:
    compute_discount(1000, "vip")
except ValueError as e:
    print("Handled gracefully →", e)

In [ ]:
# Engineers work with raw files constantly — reading & writing without pandas
# TXT
with open("data/server_logs.txt") as f:
    first_lines = [next(f) for _ in range(3)]
print("".join(first_lines))

# JSON
import json
with open("data/customers_api.json") as f:
    payload = json.load(f)
print("API status:", payload["status"], "| records:", payload["count"])
print("First record:", payload["data"][0])

## Regular Expressions (Regex) — Pattern Matching for Text (≈25 min)
Regex lets you **find, validate, and extract** patterns in text — emails, phone numbers, log entries, IDs. The `re` module is built into Python, and pandas has `.str.extract()` / `.str.contains()` built on top of it.

**Cheat sheet:**
| Pattern | Matches |
|---|---|
| `\d` | a digit (0–9) |
| `\w` | a word character (letter, digit, `_`) |
| `\s` | whitespace |
| `+` / `*` | one-or-more / zero-or-more |
| `{n,m}` | between *n* and *m* repeats |
| `[abc]` | any one of a, b, c |
| `^` / `$` | start / end of string |
| `( )` | capture group (extract this part) |

In [16]:
import re

# --- Extract structured fields from raw log lines ---
with open("data/server_logs.txt") as f:
    logs = f.readlines()

print("Raw log line:")
print(logs[0])

# One pattern, four capture groups: timestamp, level, status, response time
pattern = r"^([\d\-]+ [\d:]+) \[(\w+)\] client=([\d\.]+) endpoint=(\S+) status=(\d{3}) response_time=(\d+)ms"

m = re.match(pattern, logs[0])
print("\nParsed:", m.groups())

# Parse ALL lines into a DataFrame — regex turns unstructured text into a table!
import pandas as pd
parsed = [re.match(pattern, line).groups() for line in logs if re.match(pattern, line)]
log_df = pd.DataFrame(parsed, columns=["timestamp", "level", "client_ip", "endpoint", "status", "response_ms"])
log_df["response_ms"] = log_df["response_ms"].astype(int)
log_df["status"] = log_df["status"].astype(int)
log_df.head()

Raw log line:
2025-06-01 08:55:51 [ERROR] client=40.199.162.93 endpoint=/api/orders status=200 response_time=379ms


Parsed: ('2025-06-01 08:55:51', 'ERROR', '40.199.162.93', '/api/orders', '200', '379')


,timestamp,level,client_ip,endpoint,status,response_ms
0,2025-06-01 08:55:51,ERROR,40.199.162.93,/api/orders,200,379
1,2025-06-01 09:04:38,INFO,137.47.86.152,/checkout,200,636
2,2025-06-01 12:57:52,INFO,204.203.40.96,/login,200,1885
3,2025-06-01 15:42:46,INFO,21.168.194.245,/api/products,404,2003
4,2025-06-01 19:33:37,INFO,39.31.230.27,/api/products,200,350


In [ ]:
# --- Validating & extracting with regex ---
import re

emails = ["nina@dep.org.ph", "invalid-email", "juan.cruz@gmail.com", "test@", "ana_reyes@yahoo.com"]
email_pattern = r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$"

for e in emails:
    ok = "✅ valid" if re.match(email_pattern, e) else "❌ invalid"
    print(f"{e:30s} {ok}")

# Extract PH mobile numbers from free text
text = "Contact us at 09171234567 or 09989876543. Landline: 8123-4567."
print("\nMobile numbers found:", re.findall(r"09\d{9}", text))

# re.sub — find & replace (mask sensitive digits)
masked = re.sub(r"(09\d{2})\d{5}(\d{2})", r"\1*****\2", text)
print("Masked:", masked)

In [ ]:
# --- Regex inside pandas: clean a real column ---
import pandas as pd
sales = pd.read_csv("data/sales_data.csv")

# Flag invalid emails using .str.match()
valid = sales["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")
print(f"Invalid emails found: {(~valid).sum()} out of {len(sales)} rows")
sales.loc[~valid, "customer_email"].head()

## Part 3 — Data Formats & Schemas (≈20 min)
| Format | Type | Strengths | Weaknesses |
|---|---|---|---|
| **CSV** | Row, text | Universal, human-readable | No types, no schema, big files |
| **JSON** | Nested, text | APIs, flexible structure | Verbose, slow at scale |
| **TXT** | Raw text | Logs, anything | Needs parsing (regex!) |
| **Parquet** | Columnar, binary | Fast, compressed, typed | Not human-readable |

**Schema** = the contract: column names + types. Parquet stores it; CSV loses it.

In [9]:
import pandas as pd
import os

# Same data, three formats — compare size & type fidelity
tx = pd.read_parquet("data/transactions.parquet")
tx.to_csv("data/tx_copy.csv", index=False)
tx.to_json("data/tx_copy.json", orient="records")

for f in ["data/transactions.parquet", "data/tx_copy.csv", "data/tx_copy.json"]:
    print(f"{f:35s} {os.path.getsize(f)/1024:8.1f} KB")

# Schema fidelity: reload the CSV — the timestamp became a plain string!
tx_csv = pd.read_csv("data/tx_copy.csv")
print("\nParquet timestamp dtype:", tx["timestamp"].dtype)
print("CSV     timestamp dtype:", tx_csv["timestamp"].dtype, "← schema lost, must re-parse")

data/transactions.parquet               23.3 KB
data/tx_copy.csv                        62.3 KB
data/tx_copy.json                      129.4 KB

Parquet timestamp dtype: datetime64[ns]
CSV     timestamp dtype: str ← schema lost, must re-parse


C:\Users\Ostline\AppData\Local\Temp\ipykernel_45376\626601529.py:7: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tx.to_json("data/tx_copy.json", orient="records")


,order_id,order_date,region,product,quantity,unit_price,customer_email,phone,total_amount
0,ORD-00001,2025-04-13,Cebu,Cooking Oil 1L,5,562.36,customer0@outlook.ph,09216066793,2811.80
1,ORD-00002,2025-12-15,ncr,Rice 25kg,16,606.34,customer1@gmail.com,09510751046,9701.44
2,ORD-00003,2025-09-28,Iloilo,Cooking Oil 1L,1,1049.29,customer2@gmail.com,09141580226,1049.29
3,ORD-00004,2025-04-17,Davao,Sugar 1kg,14,595.07,customer3@outlook.ph,09605399561,8330.98
4,ORD-00005,2025-03-13,Iloilo,Instant Coffee,15,684.07,customer4@yahoo.com,09338836384,10261.05


## Part 4 — SQL with SQLite (≈30 min)
Every data engineer speaks SQL. SQLite is a full SQL database in a single file — perfect for practice, and it runs natively in Colab.

In [3]:
import sqlite3
import pandas as pd

# Create a database and load our tables into it
conn = sqlite3.connect("dep_warehouse.db")

sales = pd.read_csv("data/sales_data.csv")
tx = pd.read_parquet("data/transactions.parquet")
import json
with open("data/customers_api.json") as f:
    customers = pd.json_normalize(json.load(f)["data"])

sales.to_sql("sales", conn, if_exists="replace", index=False)
tx.to_sql("transactions", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

           name
0         sales
1  transactions
2     customers


In [4]:
# Core SQL: SELECT, WHERE, GROUP BY, ORDER BY, JOIN
import pandas as pd

q1 = """
SELECT region,
       COUNT(*)            AS orders,
       ROUND(SUM(total_amount), 2) AS revenue,
       ROUND(AVG(total_amount), 2) AS avg_order
FROM sales
WHERE quantity >= 5
GROUP BY region
ORDER BY revenue DESC;
"""
print(pd.read_sql(q1, conn))

q2 = """
SELECT c.city,
       COUNT(t.transaction_id) AS txn_count,
       ROUND(SUM(t.amount), 2) AS total_amount
FROM transactions t
JOIN customers c
  ON (CAST(SUBSTR(t.account_id, 4) AS INT) - 999) = c.customer_id  -- toy join key
GROUP BY c.city
ORDER BY total_amount DESC;
"""
pd.read_sql(q2, conn)

   region  orders    revenue  avg_order
0  Baguio      87  887107.61   10196.64
1  Iloilo      80  758964.36    9487.05
2   Davao      72  707219.95    9822.50
3     NCR      83  688275.21    8292.47
4    Cebu      75  579161.28    7722.15
5    ncr       12  114282.44    9523.54


,city,txn_count,total_amount
0,Davao City,110,258347.30
1,Makati,109,232801.10
2,Cebu City,117,221230.75
3,Taguig,85,200924.66
4,Quezon City,78,137185.23


## Part 5 — Building an ETL Pipeline (≈40 min)
**E**xtract → **T**ransform → **L**oad, written as **functions** so each step is testable and reusable. Add `logging` so the pipeline tells you what it's doing — `print()` doesn't scale.

In [6]:
import pandas as pd
import sqlite3
import logging
import re

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("etl")

# ---------- EXTRACT ----------
def extract(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    log.info(f"Extracted {len(df)} rows from {csv_path}")
    return df

# ---------- TRANSFORM ----------
def transform(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    df["region"] = df["region"].str.strip().str.upper()
    mask = df["unit_price"].isna()
    df.loc[mask, "unit_price"] = (df.loc[mask, "total_amount"] / df.loc[mask, "quantity"]).round(2)
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")
    df["email_valid"] = df["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")
    log.info(f"Transformed: {before} → {len(df)} rows "
             f"({before - len(df)} duplicates removed, {mask.sum()} prices repaired)")
    return df

# ---------- LOAD ----------
def load(df: pd.DataFrame, db_path: str, table: str) -> int:
    with sqlite3.connect(db_path) as conn:
        df.to_sql(table, conn, if_exists="replace", index=False)
    log.info(f"Loaded {len(df)} rows into {db_path}:{table}")
    return len(df)

# ---------- RUN THE PIPELINE ----------
raw = extract("data/sales_data.csv")
clean = transform(raw)
load(clean, "dep_warehouse.db", "sales_clean")
clean.to_parquet("data/sales_clean.parquet", index=False)   # also land it in the "lake"
log.info("✅ Pipeline complete")

2026-07-11 17:46:39,441 [INFO] Extracted 512 rows from data/sales_data.csv
2026-07-11 17:46:39,449 [INFO] Transformed: 512 → 500 rows (12 duplicates removed, 25 prices repaired)
2026-07-11 17:46:39,482 [INFO] Loaded 500 rows into dep_warehouse.db:sales_clean
2026-07-11 17:46:39,488 [INFO] ✅ Pipeline complete


## Part 6 — Ingesting Data from APIs (≈25 min)
The standard ingestion pattern: **request → check status → parse JSON → normalize → land it** (save to storage). Real pipelines also handle **pagination**, **rate limits**, and **retries**.

In [7]:
import requests
import pandas as pd
import json

def ingest_api(url: str, fallback_file: str) -> pd.DataFrame:
    """Fetch JSON from an API with a local fallback (offline-safe)."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        records = resp.json()
        print(f"✅ API OK ({resp.status_code}) — {len(records)} records from {url}")
    except Exception as e:
        print(f"⚠️ API unavailable ({e}); using fallback file")
        with open(fallback_file) as f:
            records = json.load(f)["data"]
    return pd.json_normalize(records)

users = ingest_api("https://jsonplaceholder.typicode.com/users", "data/customers_api.json")

# Land the raw ingest as Parquet — "raw zone" of a data lake
users.to_parquet("data/api_users_raw.parquet", index=False)
users.head(3)

✅ API OK (200) — 10 records from https://jsonplaceholder.typicode.com/users


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [8]:
# Pagination pattern — most real APIs return data in pages
import requests

all_posts = []
for page in range(1, 4):                       # fetch 3 pages
    r = requests.get("https://jsonplaceholder.typicode.com/posts",
                     params={"_page": page, "_limit": 10}, timeout=10)
    if r.status_code != 200:
        print("Stopping: status", r.status_code); break
    batch = r.json()
    if not batch:
        break                                   # no more data
    all_posts.extend(batch)
    print(f"Page {page}: +{len(batch)} records (total {len(all_posts)})")

Page 1: +10 records (total 10)
Page 2: +10 records (total 20)
Page 3: +10 records (total 30)


## Part 7 — Data Quality & Validation (≈25 min)
A pipeline that silently loads bad data is worse than one that fails loudly. Classic dimensions: **completeness, uniqueness, validity, consistency, timeliness**.

In [10]:
import pandas as pd

def validate(df: pd.DataFrame) -> dict:
    """Run a data quality report; return dict of check → pass/fail."""
    checks = {
        "no_nulls_in_key":      df["order_id"].notna().all(),
        "order_id_unique":      not df["order_id"].duplicated().any(),
        "quantity_positive":    (df["quantity"] > 0).all(),
        "price_positive":       (df["unit_price"] > 0).all(),
        "total_consistent":     ((df["quantity"] * df["unit_price"]).round(2)
                                  .sub(df["total_amount"]).abs() < 0.05).all(),
        "valid_regions":        df["region"].isin(["NCR","CEBU","DAVAO","ILOILO","BAGUIO"]).all(),
        "dates_in_range":       df["order_date"].between("2025-01-01", "2025-12-31").all(),
    }
    return checks

clean = pd.read_parquet("data/sales_clean.parquet")
report = validate(clean)

print("DATA QUALITY REPORT")
print("=" * 40)
failed = 0
for check, passed in report.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL':8s} {check}")
    failed += (not passed)
print("=" * 40)
if failed:
    raise ValueError(f"{failed} data quality check(s) failed — blocking load!")
print("All checks passed — safe to load downstream. 🚀")

DATA QUALITY REPORT
✅ PASS   no_nulls_in_key
✅ PASS   order_id_unique
✅ PASS   quantity_positive
✅ PASS   price_positive
✅ PASS   total_consistent
✅ PASS   valid_regions
✅ PASS   dates_in_range
All checks passed — safe to load downstream. 🚀


## Part 8 — Orchestration Concepts (≈15 min)
Production pipelines are **DAGs** (Directed Acyclic Graphs) of tasks, run on a schedule by tools like **Apache Airflow** or **AWS Step Functions**. We can simulate the core idea — ordered tasks with dependencies and failure handling — in plain Python.

In [14]:
import logging, time
log = logging.getLogger("orchestrator")

# A mini "DAG": each task depends on the previous one succeeding
def task(name, fn):
    log.info(f"▶ starting task: {name}")
    t0 = time.time()
    try:
        result = fn()
        log.info(f"✔ finished {name} in {time.time()-t0:.2f}s")
        return result
    except Exception as e:
        log.error(f"✖ task {name} FAILED: {e}")
        raise   # fail fast — downstream tasks must not run on bad data

import pandas as pd
pipeline_state = {}

task("extract",   lambda: pipeline_state.update(raw=extract("data/sales_data.csv")))
task("transform", lambda: pipeline_state.update(clean=transform(pipeline_state["raw"])))
task("validate",  lambda: validate(pipeline_state["clean"]))
task("load",      lambda: load(pipeline_state["clean"], "dep_warehouse.db", "sales_daily"))

print("""
In production this becomes an Airflow DAG:
  extract >> transform >> validate >> load
scheduled e.g. daily at 02:00, with retries, alerts, and backfills.""")

2026-07-11 18:51:42,526 [INFO] ▶ starting task: extract
2026-07-11 18:51:42,531 [INFO] Extracted 512 rows from data/sales_data.csv
2026-07-11 18:51:42,532 [INFO] ✔ finished extract in 0.00s
2026-07-11 18:51:42,533 [INFO] ▶ starting task: transform
2026-07-11 18:51:42,540 [INFO] Transformed: 512 → 500 rows (12 duplicates removed, 25 prices repaired)
2026-07-11 18:51:42,541 [INFO] ✔ finished transform in 0.01s
2026-07-11 18:51:42,541 [INFO] ▶ starting task: validate
2026-07-11 18:51:42,544 [INFO] ✔ finished validate in 0.00s
2026-07-11 18:51:42,546 [INFO] ▶ starting task: load
2026-07-11 18:51:42,572 [INFO] Loaded 500 rows into dep_warehouse.db:sales_daily
2026-07-11 18:51:42,573 [INFO] ✔ finished load in 0.03s



In production this becomes an Airflow DAG:
  extract >> transform >> validate >> load
scheduled e.g. daily at 02:00, with retries, alerts, and backfills.


---
# 🧪 HANDS-ON LAB (4 hours)

### Block A — Files, Formats & Regex (⭐ 45 min)
1. Convert `data/customers_api.json` to both CSV and Parquet; compare file sizes.
2. Parse `data/server_logs.txt` with regex into a DataFrame; save it as `data/logs.parquet`.
3. From the parsed logs: what % of requests are errors (status ≥ 400), and which `client_ip` appears most often?

In [122]:
# Block A — your code here
# Convert `data/customers_api.json` to both CSV and Parquet; compare file sizes.
import pandas as pd
import os
import json

with open("data/customers_api.json") as f:
    payload = json.load(f)
customers = pd.json_normalize(payload["data"])
customers.to_csv("data/customers_api_copy.csv", index=False)
customers.to_parquet("data/customers_api_copy.parquet", index=False)

for f in ["data/customers_api.json", "data/customers_api_copy.csv", "data/customers_api_copy.parquet"]:
    print(f"{f:35s} {os.path.getsize(f)/1024:8.1f} KB")

data/customers_api.json                 13.9 KB
data/customers_api_copy.csv              4.3 KB
data/customers_api_copy.parquet          6.4 KB


In [ ]:
# Parse `data/server_logs.txt` with regex into a DataFrame; save it as `data/logs.parquet`.
import re

# Extract
with open("data/server_logs.txt") as f:
    logs = f.readlines()

# Define pattern
pattern = r"^([\d\-]+ [\d:]+) \[(\w+)\] client=([\d\.]+) endpoint=(\S+) status=(\d{3}) response_time=(\d+)ms"

# Parse
parsed = [re.match(pattern, line).groups() for line in logs if re.match(pattern, line)]
log_df = pd.DataFrame(parsed, columns=["timestamp", "level", "client_ip", "endpoint", "status", "response_ms"])
log_df["response_ms"] = log_df["response_ms"].astype(int)
log_df["status"] = log_df["status"].astype(int)
log_df["timestamp"] = pd.to_datetime(log_df["timestamp"])

log_df.to_parquet("data/logs.parquet")                                                                                                                                                                                        

In [125]:
# 3. From the parsed logs: what % of requests are errors (status ≥ 400)
error_percent = ((log_df["status"] >= 400).sum() / len(log_df) * 100).round(2)
print(f"Error percentage: {error_percent}%")

Error percentage: 19.67%


In [115]:
# 3. Which `client_ip` appears most often?
top_client_ip = (log_df
    .groupby("client_ip")
    .agg(count=("client_ip", "count"))
    .sort_values(by="count", ascending=False)
    .head(1))
top_client_ip

,count
client_ip,
10.70.170.206,1


### Block B — SQL (⭐⭐ 60 min)
Using `dep_warehouse.db`:
1. Top 3 products by revenue **per region** (hint: window functions `ROW_NUMBER() OVER (PARTITION BY ...)` or a subquery).
2. Monthly transaction totals from the `transactions` table (`strftime('%Y-%m', timestamp)`).
3. Which accounts have **more than 12 transactions**? List them with their total amount.
4. Load your parsed logs table into SQLite and find the 5 slowest endpoints by average response time.

In [288]:
# Block B — your code here
import sqlite3
import pandas as pd
import json

# Create a database and load our tables into it
conn = sqlite3.connect("dep_warehouse.db")

sales = pd.read_parquet("data/sales_clean.parquet")
tx = pd.read_parquet("data/transactions.parquet")
with open("data/customers_api.json") as f:
    customers = pd.json_normalize(json.load(f)["data"])

sales.to_sql("sales", conn, if_exists="replace", index=False)
tx.to_sql("transactions", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

q1 = """
WITH ranked AS (
    SELECT
        region,
        product,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY SUM(total_amount) DESC) AS rnk
    FROM sales
    GROUP BY region, product
)
SELECT region, product, revenue
FROM ranked
WHERE rnk <= 3
ORDER BY region, rnk;
"""
print(pd.read_sql(q1, conn))


    region          product    revenue
0   BAGUIO   Cooking Oil 1L  195046.13
1   BAGUIO   Instant Coffee  174214.59
2   BAGUIO  Canned Sardines  148194.11
3     CEBU   Cooking Oil 1L  151400.66
4     CEBU  Canned Sardines  123259.54
5     CEBU        Sugar 1kg  102760.73
6    DAVAO        Rice 25kg  143982.30
7    DAVAO  Canned Sardines  137424.87
8    DAVAO   Cooking Oil 1L  134891.22
9   ILOILO   Instant Coffee  204484.65
10  ILOILO   Cooking Oil 1L  148913.25
11  ILOILO        Rice 25kg  131637.25
12     NCR        Sugar 1kg  168479.71
13     NCR     Laundry Soap  158681.40
14     NCR        Rice 25kg  144525.31


In [205]:
# 2. Monthly transaction totals from the `transactions` table (`strftime('%Y-%m', timestamp)`).
q2 = """
    SELECT 
        strftime('%Y-%m', timestamp) AS month,
        ROUND(SUM(amount), 2) AS total_amount
    FROM transactions
    GROUP BY month
    ORDER BY month
"""
print(pd.read_sql(q2, conn))

      month  total_amount
0   2025-01     124767.66
1   2025-02      93545.93
2   2025-03     149146.18
3   2025-04     140465.53
4   2025-05     153871.11
5   2025-06     132900.23
6   2025-07     143391.45
7   2025-08     127100.05
8   2025-09     131134.93
9   2025-10     159536.36
10  2025-11     179888.89
11  2025-12     108507.47


In [209]:
# 3. Which accounts have **more than 12 transactions**? List them with their total amount.
#transaction_id account_id transaction_type   amount     channel
q3 = """
    SELECT 
        account_id,
        ROUND(SUM(amount), 2) AS total_amount,
        COUNT(transaction_id) AS transaction_count
    FROM transactions
    GROUP BY account_id
    HAVING transaction_count > 12
    ORDER BY transaction_count
"""
print(pd.read_sql(q3, conn))

  account_id  total_amount  transaction_count
0    ACC1011       9427.91                 13
1    ACC1050      24675.17                 13
2    ACC1072      34354.99                 13
3    ACC1075      15672.92                 13
4    ACC1007      24658.79                 14
5    ACC1022      29353.15                 15
6    ACC1033      25067.01                 15


In [252]:
# 4. Load your parsed logs table into SQLite and find the 5 slowest endpoints by average response time.
# Create a database and load our table into it
logs = pd.read_parquet("data/logs.parquet")

logs.to_sql("logs", conn, if_exists="replace", index=False)

# Query
q4 = """
    SELECT 
        endpoint,
        ROUND(AVG(response_ms), 2) AS average_rt
    FROM logs
    GROUP BY endpoint
    ORDER BY average_rt DESC
    LIMIT 5
"""
print(pd.read_sql(q4, conn))

         endpoint  average_rt
0  /api/customers     1413.59
1       /checkout     1281.46
2          /login     1258.23
3   /api/products     1224.53
4     /api/orders     1216.66


### Block C — Build Your Own ETL (⭐⭐⭐ 75 min)
Build a pipeline `etl_transactions()` that:
1. **Extracts** `data/transactions.parquet`
2. **Transforms**: adds `amount_bucket` (small < ₱1k ≤ medium < ₱10k ≤ large), adds `txn_hour` from timestamp, filters out negative/zero amounts
3. **Validates**: unique `transaction_id`, all amounts positive, timestamps within 2025 — raise on failure
4. **Loads** into SQLite table `transactions_curated` AND writes a Parquet copy
5. Uses `logging` (not print) throughout, and wraps steps in the `task()` orchestrator pattern.

In [294]:
# Block C — your code here
import pandas as pd
import sqlite3
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("etl")

# Extracts data/transactions.parquet
def extract(parquet_path: str) -> pd.DataFrame:
    df = pd.read_parquet(parquet_path)
    log.info(f"Extracted {len(df)} rows from {parquet_path}")
    return df

# Transforms: adds amount_bucket (small < ₱1k ≤ medium < ₱10k ≤ large), adds txn_hour from timestamp, filters out negative/zero amounts
# Columns: transaction_id|account_id|transaction_type|amount|channel|timestamp|is_flagged  
def transform(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    after_duplicates = len(df)
    df["transaction_type"] = df["transaction_type"].str.strip().str.upper()
    df = df.loc[df["amount"] > 0]
    after_filter = len(df)
    df["amount_bucket"] = pd.cut(df["amount"], bins=[0, 1000, 10000, float("inf")], labels=["small", "medium", "large"])
    df["channel"] = df["channel"].str.strip().str.upper()
    df["txn_hour"] = df["timestamp"].dt.hour
    log.info(f"Transformed: {before} → {len(df)} rows "
             f"({before - after_duplicates} duplicates removed, {after_duplicates - after_filter} negative/zero amounts dropped.)")
    return df

# Validates: unique `transaction_id`, all amounts positive, timestamps within 2025 — raise on failure
def validate(df: pd.DataFrame) -> dict:
    checks = {
        "unique_transaction_id":      not df["transaction_id"].duplicated().any(),
        "amount_positive":            (df["amount"] > 0).all(),
        "valid_tx_type":              df["transaction_type"].isin(["DEPOSIT", "PAYMENT", "WITHDRAWAL", "TRANSFER"]).all(),
        "valid_channel":              df["channel"].isin(["MOBILE_APP", "ATM", "BRANCH", "ONLINE"]).all(),
        "dates_in_range":             df["timestamp"].between("2025-01-01", "2025-12-31").all(),
    }
    return checks

def report(df: pd.DataFrame):
    report = validate(df)
    print("\nDATA QUALITY REPORT")
    print("=" * 40)
    failed = 0
    for check, passed in report.items():
        print(f"{'✅ PASS' if passed else '❌ FAIL':8s} {check}")
        failed += (not passed)
    print("=" * 40)
    if failed:
        raise ValueError(f"{failed} data quality check(s) failed — blocking load!")
    log.info("All checks passed. Safe to load downstream.")

# Loads into SQLite table `transactions_curated` AND writes a Parquet copy
def load(df: pd.DataFrame, db_path: str, table: str) -> int:
    with sqlite3.connect(db_path) as conn:
        df.to_sql(table, conn, if_exists="replace", index=False)
    log.info(f"Loaded {len(df)} rows into {db_path}:{table}")
    return len(df)

def task(name, fn):
    log.info(f"▶ starting task: {name}")
    t0 = time.time()
    try:
        result = fn()
        log.info(f"✔ finished {name} in {time.time()-t0:.2f}s")
        return result
    except Exception as e:
        log.error(f"✖ task {name} FAILED: {e}")
        raise   # fail fast — downstream tasks must not run on bad data

# ---------- RUN THE PIPELINE ----------
pipeline_state = {}
task("extract",   lambda: pipeline_state.update(raw=extract("data/transactions.parquet")))
task("transform", lambda: pipeline_state.update(clean=transform(pipeline_state["raw"])))
task("validate",  lambda: report(pipeline_state["clean"]))
task("load",      lambda: load(pipeline_state["clean"], "dep_warehouse.db", "transactions_curated"))
pipeline_state["clean"].to_parquet("data/transactions_curated.parquet", index=False)
log.info("✅ Pipeline complete")

2026-07-13 03:07:48,212 [INFO] ▶ starting task: extract
2026-07-13 03:07:48,218 [INFO] Extracted 800 rows from data/transactions.parquet
2026-07-13 03:07:48,219 [INFO] ✔ finished extract in 0.01s
2026-07-13 03:07:48,219 [INFO] ▶ starting task: transform
2026-07-13 03:07:48,228 [INFO] Transformed: 800 → 800 rows (0 duplicates removed, 0 negative/zero amounts dropped.)
2026-07-13 03:07:48,230 [INFO] ✔ finished transform in 0.01s
2026-07-13 03:07:48,231 [INFO] ▶ starting task: validate
2026-07-13 03:07:48,235 [INFO] All checks passed. Safe to load downstream.
2026-07-13 03:07:48,236 [INFO] ✔ finished validate in 0.00s
2026-07-13 03:07:48,237 [INFO] ▶ starting task: load
2026-07-13 03:07:48,267 [INFO] Loaded 800 rows into dep_warehouse.db:transactions_curated
2026-07-13 03:07:48,268 [INFO] ✔ finished load in 0.03s
2026-07-13 03:07:48,273 [INFO] ✅ Pipeline complete



DATA QUALITY REPORT
✅ PASS   unique_transaction_id
✅ PASS   amount_positive
✅ PASS   valid_tx_type
✅ PASS   valid_channel
✅ PASS   dates_in_range


### Block D — Mini-Capstone: Multi-Source Pipeline (⭐⭐⭐ 60 min)
**Scenario:** Management wants a single `daily_summary` table combining all three sources.

Build a pipeline that:
1. Ingests the customers API (with fallback), the sales CSV, and the transactions Parquet
2. Produces one summary table: per city → number of customers, total sales revenue (via the toy customer_id join from earlier), and total transaction amount
3. Runs at least 4 data quality checks before loading
4. Loads the result into SQLite and exports `data/daily_summary.parquet`
5. Ends with a markdown cell: a 3-sentence "runbook" note describing what the pipeline does and how to rerun it.

In [296]:
# Block D — your capstone here
# 1. Ingests the customers API (with fallback), the sales CSV, and the transactions Parquet
import requests
import pandas as pd
import json

def ingest_customers(url: str, fallback_file: str) -> pd.DataFrame:
    #Fetch JSON from an API with a local fallback (offline-safe).
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        records = resp.json()
        log.info(f"✅ API OK ({resp.status_code}) — {len(records)} records from {url}")
    except Exception as e:
        log.warning(f"⚠️ API unavailable ({e}); using fallback file")
        with open(fallback_file) as f:
            records = json.load(f)["data"]
        log.info(f"Ingested {len(records)} records from {fallback_file}")
    return pd.json_normalize(records)

def extract_sales(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    log.info(f"Extracted {len(df)} rows from {csv_path}")
    return df

def extract_transactions(parquet_path: str) -> pd.DataFrame:
    df = pd.read_parquet(parquet_path)
    log.info(f"Extracted {len(df)} rows from {parquet_path}")
    return df

customers = ingest_customers("#https://jsonplaceholder.typicode.com/users", "data/customers_api.json")
sales = extract_sales("data/sales_clean.csv")
tx = extract_transactions("data/transactions_curated.parquet")

# 2. Produces one summary table: per city → number of customers, total sales revenue (via the toy customer_id join from earlier), and total transaction amount
conn = sqlite3.connect("dep_warehouse.db")

sales.to_sql("sales", conn, if_exists="replace", index=False)
tx.to_sql("transactions", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

summary = """
SELECT
    c.city,
    COUNT(DISTINCT c.customer_id)        AS num_customers,
    ROUND(SUM(s.total_amount), 2)        AS total_sales_revenue,
    ROUND(SUM(t.amount), 2)              AS total_tx_amount
FROM customers c
LEFT JOIN transactions t
    ON (CAST(SUBSTR(t.account_id, 4) AS INT) - 999) = c.customer_id
LEFT JOIN sales s
    ON CAST(SUBSTR(s.order_id, 5) AS INT) = CAST(SUBSTR(t.transaction_id, 4) AS INT)
GROUP BY c.city
ORDER BY total_tx_amount DESC;
"""
summary_df = pd.read_sql(summary, conn)
print(f'\n{summary_df}')

# 3. Runs at least 4 data quality checks before loading
def validate(df: pd.DataFrame) -> dict:
    checks = {
        "valid_city":                     df["city"].notna().all(),
        "valid_customers":                (df["num_customers"] >= 0).all(),
        "valid_total_sales_revenue":      (df["total_sales_revenue"] >= 0).all(),
        "valid_total_tx_amount":          (df["total_tx_amount"] >= 0).all()
    }
    return checks

def report (df: pd.DataFrame):
    report = validate(df)
    print("\nDATA QUALITY REPORT")
    print("=" * 40)
    failed = 0
    for check, passed in report.items():
        print(f"{'✅ PASS' if passed else '❌ FAIL':8s} {check}")
        failed += (not passed)
    print("=" * 40)
    if failed:
        raise ValueError(f"{failed} data quality check(s) failed — blocking load!")
    print("All checks passed. Safe to load downstream.")

report (summary_df)

# 4. Loads the result into SQLite and exports `data/daily_summary.parquet`
def load(df: pd.DataFrame, conn, table: str) -> int:
    df.to_sql(table, conn, if_exists="replace", index=False)
    log.info(f"Loaded {len(df)} rows into {table}")
    return len(df)

load(summary_df, conn, "summary")
summary_df.to_parquet("data/daily_summary.parquet", index=False)



2026-07-13 03:22:17,026 [WARNING] ⚠️ API unavailable (No connection adapters were found for '#https://jsonplaceholder.typicode.com/users'); using fallback file
2026-07-13 03:22:17,028 [INFO] Ingested 60 records from data/customers_api.json
2026-07-13 03:22:17,035 [INFO] Extracted 500 rows from data/sales_clean.csv
2026-07-13 03:22:17,042 [INFO] Extracted 800 rows from data/transactions_curated.parquet
2026-07-13 03:22:17,167 [INFO] Loaded 5 rows into summary



          city  num_customers  total_sales_revenue  total_tx_amount
0   Davao City             13            561649.34        258347.30
1       Makati             12            487969.56        232801.10
2    Cebu City             16            581285.82        221230.75
3       Taguig              9            343436.85        200924.66
4  Quezon City             10            456510.55        137185.23

DATA QUALITY REPORT
✅ PASS   valid_city
✅ PASS   valid_customers
✅ PASS   valid_total_sales_revenue
✅ PASS   valid_total_tx_amount
All checks passed. Safe to load downstream.


This pipeline ingests customer, sales, and transaction data from three sources and produces a `daily_summary` table with per-city customer count, sales revenue, and transaction totals. Transactions are linked to customers by extracting the numeric portion of `account_id` (ACC1000-ACC1099) and subtracting 999 to match `customer_id`, and sales are linked to transactions through their numeric identifiers (`order_id` and `transaction_id`) because no shared foreign key exists between the two tables in this toy dataset.

**To rerun**: ensure `data/customers_api.json`, `data/sales_clean.csv`, and `data/transactions_curated.parquet` are present, then execute Block D top to bottom.

**Note**: The customer API URL is intentionally unreachable so the local fallback file `data/customers_api.json` is always used. 

---
---
### © Data Engineering Pilipinas (DEP). All rights reserved.
*This course material was created by and is the property of **Data Engineering Pilipinas**. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.*